In [1]:
!pip install -q --upgrade wandb

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from PIL import Image, ImageFilter
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import wandb
import random
import cv2

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_api")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 74.7 MB/s eta 0:00:00:00:0100:01
Device: cuda


In [2]:
config = {
    "data_dir": Path("/kaggle/input/competitions/jaguar-re-id"),
    "checkpoint_dir": Path("checkpoints"),
    "megadescriptor_model": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 25,
    "patience": 7,
    "val_split": 0.2,
    "dropout": 0.3,
    "seed": SEED,
}

config["checkpoint_dir"].mkdir(exist_ok=True)

# Load data
train_df = pd.read_csv(config["data_dir"] / "train.csv")
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['ground_truth'])
num_classes = len(label_encoder.classes_)

train_data, val_data = train_test_split(
    train_df,
    test_size=config["val_split"],
    random_state=config["seed"],
    stratify=train_df['ground_truth']
)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Classes: {num_classes}")

Train: 1516 | Val: 379 | Classes: 31


In [7]:
preprocess = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_embeddings(model, image_paths, batch_size=32, desc="Extracting"):
    model.eval()
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=desc):
        batch_paths = image_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                tensors.append(preprocess(img))
            except:
                tensors.append(torch.zeros(3, config["input_size"],
                                           config["input_size"]))
        batch = torch.stack(tensors).to(device)
        embeddings.append(model(batch).cpu().numpy())
    return np.vstack(embeddings)


# Load MegaDescriptor
print("Loading MegaDescriptor...")
megadescriptor = timm.create_model(
    config["megadescriptor_model"], pretrained=True)
megadescriptor.eval()
megadescriptor.to(device)

with torch.no_grad():
    dummy = torch.randn(1, 3, config["input_size"],
                        config["input_size"]).to(device)
    megadescriptor_dim = megadescriptor(dummy).shape[1]

print(f"MegaDescriptor loaded | Dim: {megadescriptor_dim}")

# Paths
round1_path = config["data_dir"]
round2_path = Path("/kaggle/input/competitions/round-2-jaguar-reidentification-challenge")

cache_dir = Path("embeddings")
cache_dir.mkdir(exist_ok=True)

# Use same train/val split for both datasets
train_filenames = train_data['filename'].values
val_filenames = val_data['filename'].values

# Round 1 embeddings
r1_train_cache = cache_dir / "r1_train.npz"
r1_val_cache = cache_dir / "r1_val.npz"

if r1_train_cache.exists():
    r1_train_emb = np.load(r1_train_cache)["embeddings"]
    print(f"Loaded Round 1 train embeddings: {r1_train_emb.shape}")
else:
    print("Extracting Round 1 train embeddings...")
    paths = [round1_path / "train/train" / f for f in train_filenames]
    r1_train_emb = extract_embeddings(megadescriptor, paths,
                                       desc="R1 train")
    np.savez_compressed(r1_train_cache, embeddings=r1_train_emb)
    print(f"Saved: {r1_train_emb.shape}")

if r1_val_cache.exists():
    r1_val_emb = np.load(r1_val_cache)["embeddings"]
    print(f"Loaded Round 1 val embeddings: {r1_val_emb.shape}")
else:
    print("Extracting Round 1 val embeddings...")
    paths = [round1_path / "train/train" / f for f in val_filenames]
    r1_val_emb = extract_embeddings(megadescriptor, paths,
                                     desc="R1 val")
    np.savez_compressed(r1_val_cache, embeddings=r1_val_emb)
    print(f"Saved: {r1_val_emb.shape}")

# Round 2 embeddings
r2_train_cache = cache_dir / "r2_train.npz"
r2_val_cache = cache_dir / "r2_val.npz"

if r2_train_cache.exists():
    r2_train_emb = np.load(r2_train_cache)["embeddings"]
    print(f"Loaded Round 2 train embeddings: {r2_train_emb.shape}")
else:
    print("Extracting Round 2 train embeddings...")
    paths = [round2_path / "train" / f for f in train_filenames]
    r2_train_emb = extract_embeddings(megadescriptor, paths,
                                       desc="R2 train")
    np.savez_compressed(r2_train_cache, embeddings=r2_train_emb)
    print(f"Saved: {r2_train_emb.shape}")

if r2_val_cache.exists():
    r2_val_emb = np.load(r2_val_cache)["embeddings"]
    print(f"Loaded Round 2 val embeddings: {r2_val_emb.shape}")
else:
    print("Extracting Round 2 val embeddings...")
    paths = [round2_path / "train" / f for f in val_filenames]
    r2_val_emb = extract_embeddings(megadescriptor, paths,
                                     desc="R2 val")
    np.savez_compressed(r2_val_cache, embeddings=r2_val_emb)
    print(f"Saved: {r2_val_emb.shape}")

print("\nAll embeddings ready")

Loading MegaDescriptor...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

MegaDescriptor loaded | Dim: 1536
Extracting Round 1 train embeddings...


R1 train:   0%|          | 0/48 [00:00<?, ?it/s]

...

Saved: (379, 1536)
Extracting Round 2 train embeddings...


R2 train:   0%|          | 0/48 [00:00<?, ?it/s]

Saved: (1516, 1536)
Extracting Round 2 val embeddings...


R2 val:   0%|          | 0/12 [00:00<?, ?it/s]

Saved: (379, 1536)

All embeddings ready


In [8]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class EmbeddingProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=256, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
        )

    def forward(self, x):
        return self.network(x)

    def get_embeddings(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


class CombinedLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.arcface_weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.arcface_weight)
        self.margin = 0.5
        self.scale = 64.0

    def arcface_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.arcface_weight, dim=1)
        cosine = F.linear(embeddings, weight).clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.scale * torch.cos(theta + one_hot * self.margin)
        return F.cross_entropy(output, labels)

    def triplet_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist = torch.cdist(embeddings, embeddings, p=2)
        pos_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        neg_mask = ~pos_mask
        pos_mask.fill_diagonal_(False)
        hardest_pos = (dist * pos_mask.float()).max(dim=1)[0]
        hardest_neg = (dist + 1e6 * (~neg_mask).float()).min(dim=1)[0]
        return F.relu(hardest_pos - hardest_neg + 0.3).mean()

    def forward(self, embeddings, labels):
        return (self.alpha * self.arcface_loss(embeddings, labels) +
                (1 - self.alpha) * self.triplet_loss(embeddings, labels))


def compute_validation_map(model, val_embeddings, val_labels):
    model.eval()
    with torch.no_grad():
        val_tensor = torch.FloatTensor(val_embeddings).to(device)
        finetuned_emb = model.get_embeddings(val_tensor).cpu().numpy()

    sim_matrix = cosine_similarity(finetuned_emb)
    np.fill_diagonal(sim_matrix, -1)

    identity_aps = {}
    for query_idx in range(len(val_labels)):
        query_label = val_labels[query_idx]
        similarities = sim_matrix[query_idx]
        is_match = (val_labels == query_label).astype(int)
        is_match[query_idx] = 0
        sorted_indices = np.argsort(-similarities)
        sorted_matches = is_match[sorted_indices]
        n_positives = sorted_matches.sum()
        if n_positives == 0:
            continue
        cumsum = np.cumsum(sorted_matches)
        precision_at_k = cumsum / np.arange(1, len(sorted_matches) + 1)
        ap = np.sum(precision_at_k * sorted_matches) / n_positives
        if query_label not in identity_aps:
            identity_aps[query_label] = []
        identity_aps[query_label].append(ap)

    return np.mean([np.mean(aps) for aps in identity_aps.values()])


print("Model, loss and validation ready")

Model, loss and validation ready


In [9]:
experiments = [
    {
        "name": "bg-with-background-r1",
        "train_emb": r1_train_emb,
        "val_emb": r1_val_emb,
        "description": "Original images with background (Round 1)"
    },
    {
        "name": "bg-removed-background-r2",
        "train_emb": r2_train_emb,
        "val_emb": r2_val_emb,
        "description": "Background removed via SAM3 (Round 2)"
    },
]

val_labels = val_data['ground_truth'].values
results = []

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running: {exp['name']}")
    print(f"Description: {exp['description']}")
    print(f"{'='*60}")

    # Reset seeds
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    # Datasets
    train_dataset = EmbeddingDataset(
        exp["train_emb"],
        train_data['label_encoded'].values
    )
    val_dataset = EmbeddingDataset(
        exp["val_emb"],
        val_data['label_encoded'].values
    )

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"],
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"],
                            shuffle=False, num_workers=0)

    # Model and loss
    model = EmbeddingProjection(
        input_dim=megadescriptor_dim,
        hidden_dim=config["hidden_dim"],
        output_dim=config["embedding_dim"],
        dropout=config["dropout"]
    ).to(device)

    loss_fn = CombinedLoss(config["embedding_dim"], num_classes).to(device)

    all_params = list(model.parameters()) + list(loss_fn.parameters())
    optimizer = torch.optim.AdamW(all_params,
                                  lr=config["learning_rate"],
                                  weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    # W&B
    wandb.login(key=os.environ["WANDB_API_KEY"])
    wandb.init(
        project="jaguar-reid-mishank",
        name=exp["name"],
        config={
            **config,
            "intervention": exp["name"],
            "description": exp["description"],
            "loss_function": "combined-arcface-triplet",
            "num_parameters": sum(p.numel() for p in model.parameters()),
            "num_classes": num_classes,
        }
    )

    best_val_loss = float('inf')
    best_val_map = 0.0
    patience_counter = 0
    best_epoch = 0

    for epoch in range(config["num_epochs"]):
        # Train
        model.train()
        loss_fn.train()
        train_loss = 0

        for embeddings, labels in tqdm(train_loader,
                                       desc=f"Epoch {epoch+1}",
                                       leave=False):
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            projected = model(embeddings)
            loss = loss_fn(projected, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validate
        model.eval()
        loss_fn.eval()
        val_loss = 0

        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(device), labels.to(device)
                projected = model(embeddings)
                loss = loss_fn(projected, labels)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_map = compute_validation_map(model, exp["val_emb"], val_labels)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_map": val_map,
            "learning_rate": current_lr,
        })

        print(f"Epoch {epoch+1:2d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val mAP: {val_map:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_map = val_map
            best_epoch = epoch + 1
            patience_counter = 0
            checkpoint_path = config["checkpoint_dir"] / f"{exp['name']}_best.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_map": best_val_map,
                "val_loss": best_val_loss,
            }, checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= config["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break

    wandb.log({
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
    })
    wandb.finish()

    results.append({
        "intervention": exp["name"],
        "description": exp["description"],
        "best_val_map": round(best_val_map, 4),
        "best_val_loss": round(best_val_loss, 4),
    })

    print(f"Done: {exp['name']} | Best Val mAP: {best_val_map:.4f}")

print("\n" + "="*60)
print("ALL INTERVENTIONS COMPLETE")
print("="*60)

# Print results table
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


Running: bg-with-background-r1
Description: Original images with background (Round 1)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jain5 (jain5-university-of-potsdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch  1 | Train Loss: 15.7189 | Val Loss: 11.0690 | Val mAP: 0.4182


Epoch 2:   0%|          | 0/48 [00:00<?, ?it/s]

...

Epoch 24 | Train Loss: 0.1110 | Val Loss: 2.8817 | Val mAP: 0.7413


Epoch 25:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 25 | Train Loss: 0.0887 | Val Loss: 2.8319 | Val mAP: 0.7404


best_epoch,▁
best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▃▄▅▆▆▆▇▇▇███████████████
best_epoch,22
best_val_loss,2.79696
best_val_map,0.74209


Done: bg-removed-background-r2 | Best Val mAP: 0.7421

ALL INTERVENTIONS COMPLETE
            intervention                               description  best_val_map  best_val_loss
   bg-with-background-r1 Original images with background (Round 1)        0.7900         2.2577
bg-removed-background-r2     Background removed via SAM3 (Round 2)        0.7421         2.7970


In [10]:
# Load test pairs
test_pairs_df = pd.read_csv(config["data_dir"] / "test.csv")
test_images = sorted(list(
    set(test_pairs_df['query_image'].unique()) |
    set(test_pairs_df['gallery_image'].unique())
))

# Extract test embeddings for Round 1 test set
test_paths_r1 = [config["data_dir"] / "test/test" / f for f in test_images]
print("Extracting Round 1 test embeddings...")
test_mega_r1 = extract_embeddings(megadescriptor, test_paths_r1)

# Load best Round 1 model
model_r1 = EmbeddingProjection(
    input_dim=megadescriptor_dim,
    hidden_dim=config["hidden_dim"],
    output_dim=config["embedding_dim"],
    dropout=config["dropout"]
).to(device)

checkpoint = torch.load(
    config["checkpoint_dir"] / "bg-with-background-r1_best.pth",
    map_location=device,
    weights_only=False
)
model_r1.load_state_dict(checkpoint["model_state_dict"])
model_r1.eval()

with torch.no_grad():
    test_tensor = torch.FloatTensor(test_mega_r1).to(device)
    test_emb_r1 = model_r1.get_embeddings(test_tensor).cpu().numpy()

img_to_emb = {fn: emb for fn, emb in zip(test_images, test_emb_r1)}

similarities = []
for _, row in tqdm(test_pairs_df.iterrows(), total=len(test_pairs_df)):
    q = img_to_emb[row['query_image']]
    g = img_to_emb[row['gallery_image']]
    similarities.append(float(np.dot(q, g)))

similarities = np.clip(similarities, 0.0, 1.0)
submission_df = pd.DataFrame({
    'row_id': test_pairs_df['row_id'],
    'similarity': similarities
})
submission_df.to_csv("submission_bg_r1.csv", index=False)
print(f"Round 1 submission saved: {len(submission_df)} rows")

from IPython.display import FileLink
FileLink('submission_bg_r1.csv')

Extracting Round 1 test embeddings...


Extracting:   0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/137270 [00:00<?, ?it/s]

Round 1 submission saved: 137270 rows


/kaggle/working/submission_bg_r1.csv